## Concatenate GO & Louvain Results

## Purpose: 

1. Take all the `GO` results produced from `./2_louvain_clustering_submission.py`. 
2. Subset to results where `p_fdr_benjamini_hochberg` <0.05
3. Save metadata about that `GO` result file
4. Concatenate all `Pandas DataFrames`. 
5. Output data as `gzip'd TSV` file. 
6. Remove all the `SLURM` and script intermediary files produced from `./2_louvain_clustering_submission.py` 

Separately, store the results from all `Louvain clusterings` in `dictionary` and then `pickle`.

In [12]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import pandas as pd
pd.set_option('display.max_rows', 1000000)
pd.set_option('display.max_columns', 1000000)
pd.set_option('display.max_colwidth', 10000000)

import glob, os, shutil, pickle

## Load Data

In [2]:
parameter_combinations = [file.split("/")[-1].split(".")[0] for file in sorted(glob.glob("/home/jve4pt/RBFOX2-Mouse-Embryonic-Heart-Analysis/outputs/pickled_networks/differential_expression_tf_networks/*", recursive=True))]

parameter_combinations

['upstream_2500_score_900',
 'upstream_2500_score_950',
 'upstream_2500_score_990',
 'upstream_5000_score_900',
 'upstream_5000_score_950',
 'upstream_5000_score_990',
 'upstream_750_score_900',
 'upstream_750_score_950',
 'upstream_750_score_990']

## Concatenation Algorithm

In [ ]:
# dataframe with all the Gene Ontology analysis results in one
meta_GO_df = pd.DataFrame()

"""
for GO results for all possible parameter combinations, iterations, and clusters, 
retrieve them via `glob`, subset to FDR p < 0.05, 
save info about parameter combination, iteration and cluster
and finally concatenate all these together

# OPTIONAL: filtering GO terms by keywords

"""



for parameter_combination in parameter_combinations:    
    for iteration in range(1000): 
                
        tmp_files = glob.glob(
            "../../../../outputs/gene_ontology_analysis/differential_expression_networks/output/{}-iter_{}/*.tsv".format(parameter_combination, iteration)
        )
        
        if len(tmp_files)==0: 
            print("{} {}".format(parameter_combination, iteration))
            
        for file in tmp_files: 
            
            cluster = file.split("-")[-1].split(".")[0]
                        
            df = pd.read_csv(file , sep="\t")
            
            df = df[df["p_fdr_bh"]<0.05].copy()
                        
#             # subset to cardiac relevant tissue 
#             df =df[(df["name"].str.contains("blood")) | (df["name"].str.contains("heart")) | (df["name"].str.contains("vascul")) | (df["name"].str.contains("circul")) | (df["name"].str.contains("cardiac"))]
            
            if df.index.size>0:
                
                df["Parameter_Combination"] = parameter_combination
                df["Iteration"] = iteration 
                df["Cluster"] = cluster
            
                meta_GO_df = pd.concat([meta_GO_df, df])



In [ ]:
meta_GO_df.index.size

meta_GO_df.head()

## Output `DataFrame` in `gzip` Format 

In [7]:
meta_GO_df.to_csv(
    "../../../../outputs/gene_ontology_analysis/differential_expression_networks/all_GO_analyses.tsv.gz", 
    sep="\t", 
    compression='gzip', 
    index=False
)

Make sure that file is properly formatted

In [9]:
tmp = pd.read_csv(
    "../../../../outputs/gene_ontology_analysis/differential_expression_networks/all_GO_analyses.tsv.gz", 
    sep="\t",
    compression='gzip',
)

tmp.index.size
tmp.head()

141906

,# GO,NS,enrichment,name,ratio_in_study,ratio_in_pop,p_uncorrected,depth,study_count,p_fdr_bh,study_items,Parameter_Combination,Iteration,Cluster
0,GO:0098637,CC,e,protein complex involved in cell-matrix adhesion,2/12,9/21965,9.829035e-06,3,2,0.017850,"lama1, vtn",upstream_2500_score_900,0,cluster_8
1,GO:0003700,MF,e,DNA-binding transcription factor activity,8/29,608/21965,8.450714e-07,2,8,0.003890,"crebrf, elf4, elk4, hoxb5, nr2f1, onecut2, rreb1, tbx3",upstream_2500_score_900,0,cluster_2
2,GO:0001228,MF,e,"DNA-binding transcription activator activity, RNA polymerase II-specific",7/29,511/21965,3.543401e-06,4,7,0.007910,"crebrf, elf4, elk4, hoxb5, onecut2, rreb1, tbx3",upstream_2500_score_900,0,cluster_2
3,GO:1990837,MF,e,sequence-specific double-stranded DNA binding,7/29,541/21965,5.155510e-06,6,7,0.007910,"elf4, elk4, hoxb5, nr2f1, onecut2, tbx3, zbtb12",upstream_2500_score_900,0,cluster_2
4,GO:0071949,MF,e,FAD binding,3/16,43/21965,3.844505e-06,6,3,0.017696,"dhcr24, sqle, xdh",upstream_2500_score_900,1,cluster_1


## Remove Original Output from Script + `SLURM`

In [10]:
if os.path.exists("../../../../outputs/gene_ontology_analysis/differential_expression_networks/output/"): 
    shutil.rmtree("../../../../outputs/gene_ontology_analysis/differential_expression_networks/output/")
    
if os.path.exists("../../../../outputs/gene_ontology_analysis/differential_expression_networks/SLURM/"):    
    shutil.rmtree("../../../../outputs/gene_ontology_analysis/differential_expression_networks/SLURM/")

## Create Dict of Louvain Results and Pickle

In [13]:
louvain_results = {}

# load all the pickled louvain community clustering results 
# into a dict such that it's organized by parameter combination and iteration

for parameter_combination in parameter_combinations:
        
    louvain_results[parameter_combination]={}
    
    for iteration in range(1000): 
        louvain_results[parameter_combination][iteration] = pickle.load(open("../../../../outputs/louvain_clustering/{}/{}-iter_{}.pkl".format(parameter_combination, parameter_combination, iteration), 'rb'))


In [18]:
pickle.dump(louvain_results, open("../../../../outputs/louvain_clustering/all_louvain_clusterings.pkl", 'wb'))

In [22]:
tmp_test = pickle.load(open("../../../../outputs/louvain_clustering/all_louvain_clusterings.pkl", 'rb'))

In [24]:
tmp_test["upstream_2500_score_900"][1][0]

{'apoa1',
 'blk',
 'car3',
 'cbfa2t3',
 'col6a3',
 'crct1',
 'edn1',
 'emcn',
 'fabp4',
 'fsd1l',
 'fyb2',
 'gata6',
 'hhex',
 'krt19',
 'lacc1',
 'lamc1',
 'ldlr',
 'lipo3',
 'lmcd1',
 'medag',
 'nppb',
 'onecut2',
 'pdp2',
 'psca',
 'psd',
 'rora',
 'sprr2h',
 'stat6',
 'sytl4',
 'tuft1',
 'vegfa'}

## Remove Extraneous Clustering Data Folders

In [20]:
folders = glob.glob("../../../../outputs/louvain_clustering/upstream_*")
folders

['../../../../outputs/louvain_clustering/upstream_750_score_950',
 '../../../../outputs/louvain_clustering/upstream_5000_score_900',
 '../../../../outputs/louvain_clustering/upstream_5000_score_950',
 '../../../../outputs/louvain_clustering/upstream_2500_score_950',
 '../../../../outputs/louvain_clustering/upstream_2500_score_990',
 '../../../../outputs/louvain_clustering/upstream_750_score_900',
 '../../../../outputs/louvain_clustering/upstream_2500_score_900',
 '../../../../outputs/louvain_clustering/upstream_5000_score_990',
 '../../../../outputs/louvain_clustering/upstream_750_score_990']

In [25]:
for folder in folders: 
    shutil.rmtree(folder)